# ハイパーパラメータをグリッドサーチで調整する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/grid_search.ipynb)

NLP クックブック 12-07 ／ テキスト分類

TF-IDF の設定や正則化の強さなど、複数のハイパーパラメータの最良の組み合わせを交差検証で探したい。

In [1]:
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

cats = ["rec.sport.baseball", "sci.space", "comp.graphics", "talk.politics.mideast"]
train = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))
test = fetch_20newsgroups(subset="test", categories=cats, remove=("headers", "footers", "quotes"))

pipe = Pipeline([("tfidf", TfidfVectorizer(stop_words="english")), ("clf", LogisticRegression(max_iter=1000))])
grid = GridSearchCV(pipe, {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__sublinear_tf": [False, True],
    "tfidf__min_df": [1, 3],
    "clf__C": [0.1, 1, 10, 100],
}, cv=3, scoring="f1_macro", n_jobs=-1)
grid.fit(train.data, train.target)

print("最良:", grid.best_params_, round(grid.best_score_, 4))
print("テスト macro-F1:", round(grid.score(test.data, test.target), 4))

最良: {'clf__C': 100, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True} 0.9125
テスト macro-F1: 0.8999


In [2]:
res = pd.DataFrame(grid.cv_results_)[["param_tfidf__ngram_range", "param_tfidf__sublinear_tf", "param_tfidf__min_df", "param_clf__C", "mean_test_score", "std_test_score"]]
res.sort_values("mean_test_score", ascending=False).head(5).round(4)

,param_tfidf__ngram_range,param_tfidf__sublinear_tf,param_tfidf__min_df,param_clf__C,mean_test_score,std_test_score
27,"(1, 2)",True,1,100.0,0.9125,0.0052
17,"(1, 1)",True,1,10.0,0.9110,0.0037
19,"(1, 2)",True,1,10.0,0.9109,0.0072
25,"(1, 1)",True,1,100.0,0.9096,0.0022
18,"(1, 2)",False,1,10.0,0.9093,0.0071


- パラメータ名は `ステップ名__パラメータ名`。パイプラインごと探索することで、前処理の設定も一緒に最適化できる。
- 探索空間が広いときは `RandomizedSearchCV` や Optuna を使う。
- 上位の組み合わせのスコア差が標準偏差より小さければ、実質的に同等。